# M3 — Transformations and financial indicators

Work through the formulas, then compare them with reusable functions. Use only trailing information at signal time. Monthly macro availability is an assumed observation-month-end plus 60 days; unverified releases and revisions remain a limitation.

Run cells in order with the project environment (`uv run jupyter lab`). Paths support the project root or `notebooks/`. See [workshop order](../_docs/workshop_order.md) for dependencies and checkpoints. Numerical outputs come from actual local inputs; no synthetic data are substituted for market results.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "config/project.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config/project.yaml").exists(), "Launch from the project root or notebooks/"
from uranium_signals.config import load_config
from uranium_signals.universe import load_universe
from uranium_signals.providers import acquire
from uranium_signals.run import research_tables
cfg = load_config(ROOT / "config/project.yaml")
universe = load_universe(cfg)
tickers = set(universe.ticker) | {cfg["benchmark"], cfg["broad_benchmark"]} | set(cfg["optional_benchmarks"])


In [ ]:
prices, macro, source_metadata = acquire(cfg, tickers, offline=True)
prices, quality, targets, table = research_tables(cfg, prices, macro, universe)


## 1. Simple, log and compounded returns

For adjusted price P, simple return is `P[t] / P[t-1] - 1`. Log return is `log(P[t] / P[t-1])`. Wealth compounds through `(1 + return).cumprod()`; summing simple returns is incorrect. Daily calculations below use Cameco.

In [ ]:
ccj = prices[prices.ticker.eq("CCJ")].set_index("date").sort_index()
p = ccj.adj_close
simple = p / p.shift(1) - 1
log_return = np.log(p / p.shift(1))
wealth = (1 + simple.dropna()).cumprod()
assert np.isclose(wealth.iloc[-1], p.iloc[-1] / p.iloc[0])
assert np.isclose(np.exp(log_return.dropna().sum()), wealth.iloc[-1])
weekly_price = p.resample("W-FRI").last()
monthly_price = p.resample("ME").last()
display(pd.DataFrame({"price": p, "simple_return": simple, "log_return": log_return}).tail())


## 2. Trailing financial indicators

A moving-average distance measures deviation from a trailing trend. Annualized volatility multiplies daily standard deviation by sqrt(252). Downside deviation uses negative daily returns with zero as the threshold. Drawdown compares price with a trailing peak. These are descriptions, not causal explanations.

In [ ]:
worked = pd.DataFrame(index=p.index)
worked["momentum_63"] = p / p.shift(63) - 1
worked["ma_distance_50"] = p / p.rolling(50).mean() - 1
worked["volatility_21"] = simple.rolling(21).std() * np.sqrt(252)
worked["downside_21"] = simple.clip(upper=0).pow(2).rolling(21).mean().pow(.5) * np.sqrt(252)
worked["drawdown_126"] = p / p.rolling(126).max() - 1
worked["approx_dollar_volume_63"] = (p * ccj.volume).rolling(63).median()
display(worked.tail())
worked[["momentum_63", "drawdown_126"]].plot(title="Cameco trailing momentum and drawdown", figsize=(10, 4))
plt.show()


## 3. Monthly information alignment

Features are sampled on URA's last trading session in each completed month. Cross-sectional ranks use eligible candidates only. Relative momentum subtracts benchmark momentum. Macro changes are calculated on a regular monthly grid, then joined backward on their assumed availability date. No future values are backfilled.

Real uranium price equals nominal price divided by CPI relative to a stated base. The latest CPI base is useful for display but is excluded from model predictors.

In [ ]:
from uranium_signals.features import FEATURE_COLUMNS, macro_features
m = macro_features(macro, cfg)
display(m.tail())
assert (table.macro_available_at.dropna() <= table.loc[table.macro_available_at.notna(), "date"]).all()
latest_month = table[table.date.eq(table.date.max())]
display(latest_month[["ticker", "eligible", "ret_63", "rel_63", "momentum_rank", "macro_observation_date", "macro_available_at"]])
latest_month.set_index("ticker")["rel_63"].sort_values().plot.barh(title="Latest monthly relative momentum", figsize=(8, 5))
plt.show()


## 4. Check missingness and persist the table

Warm-up periods are expected for 200-session averages and 252-session eligibility. Do not fill them with future observations. Imputation used by a model must be fit on training data only.

In [ ]:
display(table[FEATURE_COLUMNS].isna().mean().sort_values(ascending=False).rename("missing_fraction"))
table.to_csv(ROOT / "data/processed/monthly_features.csv", index=False)
assert "forward_return" not in FEATURE_COLUMNS
assert "outperform_1m" not in FEATURE_COLUMNS
print("Feature checkpoint passed.")


## Written-result checkpoint

Explain one momentum calculation, one volatility calculation, and the date on which one macro observation first becomes usable under the proxy lag. Inspect distributions/outliers and explain why latest-revised macro data do not establish a fully point-in-time information set.